# 实验四：基于大语言模型的唐诗自动生成微调实验

## 实验流程
1. 安装依赖
2. 加载模型（Baseline）
3. Baseline 推理测试
4. 数据准备
5. LoRA 微调训练
6. 微调后推理对比

In [ ]:
# Cell 1: 安装依赖
# 主包用 --no-deps，不升级镜像自带的 torch/numpy/pandas
!pip install huggingface_hub transformers==4.37.0 datasets peft accelerate --no-deps -q

# 只装缺的运行时依赖，锁定兼容版本
!pip install "numpy<2.0" pyarrow dill multiprocess fsspec requests tqdm xxhash psutil safetensors regex tokenizers sentencepiece protobuf filelock packaging -q

In [ ]:
# Cell 2: 导入库 + 加载模型
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
import json

# 检查 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前设备: {device}")
print(f"PyTorch 版本: {torch.__version__}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 加载模型
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
print(f"\n正在加载模型: {MODEL_NAME} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    trust_remote_code=True
)

if device == "cpu":
    model = model.to("cpu")

print("模型加载完成！")
print(f"模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# Cell 3: 定义生成函数
def generate_poetry(prompt, model, max_len=80, temp=0.7, top_k=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_len,
            do_sample=True,
            temperature=temp,
            top_k=top_k,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Cell 4: 测试集 Prompt
test_prompts = [
    "床前明月光，",
    "白日依山尽，",
    "红豆生南国，",
    "空山不见人，",
    "独坐幽篁里，",
    "春眠不觉晓，",
    "千山鸟飞绝，",
    "松下问童子，",
    "功盖三分国，",
    "绿蚁新醅酒，",
    "寒雨连江夜入吴，",
    "黄河远上白云间，",
    "独在异乡为异客，"
]

In [ ]:
# Cell 5: Baseline 推理测试
print("=" * 60)
print("Baseline 模型（未微调）生成结果")
print("=" * 60)

baseline_results = []
for prompt in test_prompts:
    result = generate_poetry(prompt, model)
    baseline_results.append({"prompt": prompt, "output": result})
    print(f"\n输入: {prompt}")
    print(f"输出: {result}")
    print("-" * 40)

---
## Part 2: LoRA 微调

In [ ]:
# Cell 6: 加载训练数据
poems = []
for enc in ['utf-8', 'gbk', 'gb2312', 'gb18030']:
    try:
        with open('唐诗三百 - small.txt', 'r', encoding=enc) as f:
            poems = [line.strip() for line in f.readlines() if line.strip()]
        print(f"使用编码 {enc} 成功读取，共 {len(poems)} 首诗")
        break
    except:
        continue

if not poems:
    with open('poetry_utf8.txt', 'r', encoding='utf-8') as f:
        poems = [line.strip() for line in f.readlines() if line.strip()]
    print(f"使用 poetry_utf8.txt，共 {len(poems)} 首诗")

print("\n前5首诗:")
for i, p in enumerate(poems[:5]):
    print(f"  {i+1}. {p}")

In [ ]:
# Cell 7: 数据预处理
def preprocess_poems(poems, tokenizer, max_length=128):
    encodings = tokenizer(
        poems,
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_tensors="pt"
    )
    return Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": encodings["input_ids"].clone()
    })

train_dataset = preprocess_poems(poems, tokenizer)
print(f"训练样本数: {len(train_dataset)}")

In [ ]:
# Cell 8: 配置 LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

model_lora = get_peft_model(model, lora_config)
model_lora.print_trainable_parameters()

print(f"\nLoRA 配置:")
print(f"  r = {lora_config.r}")
print(f"  lora_alpha = {lora_config.lora_alpha}")
print(f"  target_modules = {lora_config.target_modules}")
print(f"  lora_dropout = {lora_config.lora_dropout}")

In [ ]:
# Cell 9: 配置训练参数
training_args = TrainingArguments(
    output_dir="./qwen-poetry-lora",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    fp16=(device == "cuda"),
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
)

print("训练参数:")
print(f"  学习率: {training_args.learning_rate}")
print(f"  批次大小: {training_args.per_device_train_batch_size}")
print(f"  训练轮数: {training_args.num_train_epochs}")
print(f"  fp16: {training_args.fp16}")

In [ ]:
# Cell 10: 开始训练
trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_dataset,
)

print("开始训练...")
train_result = trainer.train()

print(f"\n训练完成！")
print(f"总训练步数: {train_result.global_step}")
print(f"最终 Loss: {train_result.training_loss:.4f}")

In [ ]:
# Cell 11: 绘制 Loss 曲线
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_loss = [x['loss'] for x in logs if 'loss' in x]
steps = [x['step'] for x in logs if 'loss' in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, train_loss, 'b-', linewidth=2)
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('LoRA Fine-tuning Loss Curve')
plt.grid(True)
plt.savefig('loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Loss 曲线已保存为 loss_curve.png")

In [ ]:
# Cell 12: 保存 LoRA 权重
save_dir = "./qwen-poetry-lora-final"
model_lora.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"LoRA 权重已保存到: {save_dir}")

---
## Part 3: 推理对比

In [ ]:
# Cell 13: 重新加载基线模型 + LoRA 权重
print("重新加载基线模型...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    trust_remote_code=True
)
if device == "cpu":
    base_model = base_model.to("cpu")

print("加载 LoRA 权重...")
finetuned_model = PeftModel.from_pretrained(base_model, "./qwen-poetry-lora-final")
print("微调模型加载完成！")

In [ ]:
# Cell 14: 对比测试
print("=" * 70)
print("Baseline vs LoRA 微调 对比结果")
print("=" * 70)

comparison_results = []

for prompt in test_prompts:
    baseline_out = generate_poetry(prompt, model, max_len=60)
    lora_out = generate_poetry(prompt, finetuned_model, max_len=60)
    
    comparison_results.append({
        "prompt": prompt,
        "baseline": baseline_out,
        "lora_finetuned": lora_out
    })
    
    print(f"\n输入: {prompt}")
    print(f"  Baseline:  {baseline_out}")
    print(f"  LoRA微调:  {lora_out}")
    print("-" * 50)

with open("comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(comparison_results, f, ensure_ascii=False, indent=2)
print("\n对比结果已保存到 comparison_results.json")

In [ ]:
# Cell 15: 结果统计
print("\n" + "=" * 70)
print("实验总结")
print("=" * 70)
print(f"训练样本数: {len(poems)}")
print(f"训练轮数: {training_args.num_train_epochs}")
print(f"学习率: {training_args.learning_rate}")
print(f"LoRA rank: {lora_config.r}")
print(f"最终 Loss: {train_result.training_loss:.4f}")
print(f"\n测试 Prompt 数: {len(test_prompts)}")
print("\n对比结果见 comparison_results.json")
print("Loss 曲线见 loss_curve.png")